# 21b — Neural Model Susceptibility and FDR Correction

This notebook extends the susceptibility analysis to the neural models tested in Notebook 20.

Notebook 21 corrected the older susceptibility tables, which only included:

- LDA
- Logistic Regression
- EEGNet

This notebook adds susceptibility correlations and FDR correction for the expanded model set from Notebook 20:

- LDA
- Logistic Regression
- Current EEGNet
- Tuned EEGNet
- ShallowConvNet
- DeepConvNet
- DenseNet-style EEG CNN

The notebook tests whether raw/ERP/power features and ICA/source-structure features explain subject-level performance for these models.

Inputs:

- Subject-level model performance from Notebook 20
- Raw/ERP/power features from Notebook 17b
- ICA features from Notebook 18

Outputs:

- Raw/ERP/power susceptibility correlations for all Notebook 20 models
- ICA susceptibility correlations for all Notebook 20 models
- Benjamini-Hochberg FDR q-values
- Tables of effects surviving q ≤ 0.05 and exploratory q ≤ 0.10

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr

from oddball_eeg.io import save_dataframe
from oddball_eeg.stats import benjamini_hochberg, safe_corr, zscore_within_group
from oddball_eeg.visualization import (
    round_numeric_columns,
    save_figure,
    set_pandas_display_options,
)

set_pandas_display_options()

# ------------------------------------------------------------
# Input folders
# ------------------------------------------------------------

NEURAL_TABLE_DIR = Path("results") / "thesis_artifacts" / "neural_model_diagnostics" / "tables"

RAW_TABLE_DIR = Path("results") / "thesis_artifacts" / "multimodel_classification_susceptibility" / "tables"

ICA_TABLE_DIR = Path("results") / "thesis_artifacts" / "ica_multimodel_classification_susceptibility" / "tables"

# ------------------------------------------------------------
# Input files
# ------------------------------------------------------------

NEURAL_SUBJECT_SUMMARY_PATH = NEURAL_TABLE_DIR / "subject_level_model_summary.csv"

RAW_FEATURE_PATH = RAW_TABLE_DIR / "copied_subject_raw_eeg_features.csv"

ICA_FEATURE_PATH = ICA_TABLE_DIR / "ica_subject_features.csv"

# ------------------------------------------------------------
# Output folder
# ------------------------------------------------------------

OUT_ROOT = Path("results") / "thesis_artifacts" / "neural_model_susceptibility_fdr"
TABLE_DIR = OUT_ROOT / "tables"
FIG_DIR = OUT_ROOT / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

Q_THRESHOLDS = [0.05, 0.10]

print("Neural model subject summary:", NEURAL_SUBJECT_SUMMARY_PATH)
print("Raw feature table:", RAW_FEATURE_PATH)
print("ICA feature table:", ICA_FEATURE_PATH)
print("Output root:", OUT_ROOT)

In [ ]:
def normalize_subject_id(x) -> str:
    """
    Normalize subject ID as string while preserving leading zeros when present.
    """
    s = str(x).strip()

    if s.endswith(".0"):
        s = s[:-2]

    return s


def add_fdr_by_family(
        df: pd.DataFrame,
        *,
        p_col: str,
        family_cols: list[str],
        q_col: str,
        sig_prefix: str,
) -> pd.DataFrame:
    out = df.copy()
    out[q_col] = np.nan

    for _, idx in out.groupby(family_cols, dropna=False).groups.items():
        idx = list(idx)
        out.loc[idx, q_col] = benjamini_hochberg(out.loc[idx, p_col].values)

    for q_thr in Q_THRESHOLDS:
        if q_thr == 0.05:
            label = "005"
        elif q_thr == 0.10:
            label = "010"
        else:
            label = str(q_thr).replace(".", "")

        out[f"{sig_prefix}_fdr_{label}"] = out[q_col] <= q_thr

    return out


def add_standard_fdr_columns(
        df: pd.DataFrame,
        *,
        family_cols: list[str],
        pearson_p_col: str,
        spearman_p_col: str,
        pearson_q_col: str,
        spearman_q_col: str,
) -> pd.DataFrame:
    out = df.copy()

    if pearson_p_col in out.columns:
        out = add_fdr_by_family(
            out,
            p_col=pearson_p_col,
            family_cols=family_cols,
            q_col=pearson_q_col,
            sig_prefix=pearson_q_col,
        )

    if spearman_p_col in out.columns:
        out = add_fdr_by_family(
            out,
            p_col=spearman_p_col,
            family_cols=family_cols,
            q_col=spearman_q_col,
            sig_prefix=spearman_q_col,
        )

    return out


def top_fdr_significant(
        df: pd.DataFrame,
        *,
        q_col: str,
        effect_col: str,
        q_threshold: float = 0.05,
        n_top: int = 100,
) -> pd.DataFrame:
    out = df[df[q_col] <= q_threshold].copy()

    if len(out) == 0:
        return out

    out["abs_effect_for_sort"] = np.abs(out[effect_col])

    out = out.sort_values(
        [q_col, "abs_effect_for_sort"],
        ascending=[True, False],
    ).head(n_top)

    return out.drop(columns=["abs_effect_for_sort"], errors="ignore")


def summarize_fdr_table(
        df: pd.DataFrame,
        *,
        table_name: str,
        family_cols: list[str],
        q_col: str,
        p_col: str,
        effect_col: str,
) -> pd.DataFrame:
    rows = []

    for keys, g in df.groupby(family_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {
            "table_name": table_name,
            **dict(zip(family_cols, keys)),
            "n_tests": int(np.isfinite(g[p_col]).sum()),
            "n_fdr_05": int((g[q_col] <= 0.05).sum()),
            "n_fdr_10": int((g[q_col] <= 0.10).sum()),
            "min_p": float(np.nanmin(g[p_col])) if np.isfinite(g[p_col]).any() else np.nan,
            "min_q": float(np.nanmin(g[q_col])) if np.isfinite(g[q_col]).any() else np.nan,
            "max_abs_effect": float(np.nanmax(np.abs(g[effect_col]))) if effect_col in g.columns else np.nan,
        }

        rows.append(row)

    return pd.DataFrame(rows)


def clean_feature_list(df: pd.DataFrame, feature_cols: list[str], min_nonnull: int = 10) -> list[str]:
    cleaned = []

    for col in feature_cols:
        if col not in df.columns:
            continue

        x = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)

        if x.notna().sum() < min_nonnull:
            continue

        if x.nunique(dropna=True) <= 1:
            continue

        cleaned.append(col)

    return cleaned

In [ ]:
for path in [
    NEURAL_SUBJECT_SUMMARY_PATH,
    RAW_FEATURE_PATH,
    ICA_FEATURE_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file:\n{path}")

model_perf_df = pd.read_csv(NEURAL_SUBJECT_SUMMARY_PATH)
raw_feature_df = pd.read_csv(RAW_FEATURE_PATH)
ica_feature_df = pd.read_csv(ICA_FEATURE_PATH)

# Normalize subject IDs.
for df in [model_perf_df, raw_feature_df, ica_feature_df]:
    df["subject"] = df["subject"].map(normalize_subject_id)

# Replace inf with NaN.
model_perf_df = model_perf_df.replace([np.inf, -np.inf], np.nan)
raw_feature_df = raw_feature_df.replace([np.inf, -np.inf], np.nan)
ica_feature_df = ica_feature_df.replace([np.inf, -np.inf], np.nan)

print("Model performance rows:", len(model_perf_df))
print("Raw feature rows:", len(raw_feature_df))
print("ICA feature rows:", len(ica_feature_df))

print("\nModels in Notebook 20 summary:")
display(
    model_perf_df
    .groupby(["model", "model_family"])["subject"]
    .count()
    .reset_index(name="n_subject_recordings")
)

display(round_numeric_columns(model_perf_df.head(), 4))
display(round_numeric_columns(raw_feature_df.head(), 4))
display(round_numeric_columns(ica_feature_df.head(), 4))

In [ ]:
MERGE_KEYS = [
    "experiment_name",
    "dataset_label",
    "subject",
]

# Keep only columns needed from model performance.
model_cols = [
    "dataset_label",
    "experiment_name",
    "subject",
    "model",
    "model_family",
    "auc_mean",
    "auc_std",
    "pr_auc_mean",
    "bal_acc_mean",
    "f1_mean",
    "n_folds",
]

available_model_cols = [c for c in model_cols if c in model_perf_df.columns]

model_perf_small_df = model_perf_df[available_model_cols].copy()

raw_subject_df = raw_feature_df.copy()
ica_subject_df = ica_feature_df.copy()

raw_merged_df = model_perf_small_df.merge(
    raw_subject_df,
    on=MERGE_KEYS,
    how="inner",
    suffixes=("", "_raw_duplicate"),
)

ica_merged_df = model_perf_small_df.merge(
    ica_subject_df,
    on=MERGE_KEYS,
    how="inner",
    suffixes=("", "_ica_duplicate"),
)

save_dataframe(
    raw_merged_df,
    TABLE_DIR / "neural_models_raw_feature_merged_subject_table.csv",
)

save_dataframe(
    ica_merged_df,
    TABLE_DIR / "neural_models_ica_feature_merged_subject_table.csv",
)

print("Raw merged rows:", len(raw_merged_df))
print("ICA merged rows:", len(ica_merged_df))

print("\nRaw rows per model:")
display(
    raw_merged_df
    .groupby("model")["subject"]
    .count()
    .reset_index(name="n_subject_recordings")
)

print("\nICA rows per model:")
display(
    ica_merged_df
    .groupby("model")["subject"]
    .count()
    .reset_index(name="n_subject_recordings")
)

In [ ]:
RAW_ID_COLS = {
    "experiment_name",
    "dataset",
    "task",
    "event_preset",
    "dataset_label",
    "subject",
    "model",
    "model_family",
    "roi_channels",
    "roi_method",
}

RAW_TARGET_COLS = {
    "auc_mean",
    "auc_std",
    "pr_auc_mean",
    "bal_acc_mean",
    "f1_mean",
    "n_folds",
}

ICA_ID_COLS = {
    "experiment_name",
    "dataset",
    "task",
    "event_preset",
    "dataset_label",
    "subject",
    "model",
    "model_family",
    "montage_name",
}

ICA_TARGET_COLS = {
    "auc_mean",
    "auc_std",
    "pr_auc_mean",
    "bal_acc_mean",
    "f1_mean",
    "n_folds",
}

raw_feature_cols = [
    c for c in raw_merged_df.select_dtypes(include=[np.number]).columns
    if c not in RAW_ID_COLS
       and c not in RAW_TARGET_COLS
       and not (c.startswith("power_") and not c.startswith("log_"))
]

ica_feature_cols = [
    c for c in ica_merged_df.select_dtypes(include=[np.number]).columns
    if c not in ICA_ID_COLS
       and c not in ICA_TARGET_COLS
]

raw_feature_cols = clean_feature_list(raw_merged_df, raw_feature_cols)
ica_feature_cols = clean_feature_list(ica_merged_df, ica_feature_cols)

print("Raw/ERP/power features:", len(raw_feature_cols))
for c in raw_feature_cols:
    print(" -", c)

print("\nICA features:", len(ica_feature_cols))
for c in ica_feature_cols:
    print(" -", c)

feature_summary_df = pd.DataFrame([
    {
        "feature_family": "raw_erp_power",
        "n_features": len(raw_feature_cols),
        "features": ",".join(raw_feature_cols),
    },
    {
        "feature_family": "ica",
        "n_features": len(ica_feature_cols),
        "features": ",".join(ica_feature_cols),
    },
])

save_dataframe(
    feature_summary_df,
    TABLE_DIR / "feature_summary.csv",
)

display(feature_summary_df)

In [ ]:
def compute_susceptibility_correlations(
        df: pd.DataFrame,
        *,
        feature_cols: list[str],
        target_col: str = "auc_mean",
        family_name: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Compute:
    1. within-dataset correlations per model × dataset
    2. grand z-scored correlations per model
    3. z-scored subject table
    """

    # ------------------------------------------------------------
    # Within-dataset correlations
    # ------------------------------------------------------------
    within_rows = []

    for model, model_df in df.groupby("model"):
        for dataset_label, g in model_df.groupby("dataset_label"):
            for feature in feature_cols:
                pearson_r, pearson_p, n_p = safe_corr(
                    g[target_col],
                    g[feature],
                    method="pearson",
                )

                spearman_rho, spearman_p, n_s = safe_corr(
                    g[target_col],
                    g[feature],
                    method="spearman",
                )

                within_rows.append({
                    "feature_family": family_name,
                    "model": model,
                    "dataset_label": dataset_label,
                    "feature": feature,
                    "n": n_p,
                    "pearson_r": pearson_r,
                    "pearson_p": pearson_p,
                    "spearman_rho": spearman_rho,
                    "spearman_p": spearman_p,
                    "abs_pearson_r": abs(pearson_r) if np.isfinite(pearson_r) else np.nan,
                    "abs_spearman_rho": abs(spearman_rho) if np.isfinite(spearman_rho) else np.nan,
                })

    within_corr_df = pd.DataFrame(within_rows)

    within_corr_df = within_corr_df.sort_values(
        ["model", "dataset_label", "abs_spearman_rho"],
        ascending=[True, True, False],
    ).reset_index(drop=True)

    # ------------------------------------------------------------
    # Grand z-scored correlations
    # ------------------------------------------------------------
    z_df = df.copy()

    z_df["auc_z"] = (
        z_df
        .groupby(["model", "dataset_label"])[target_col]
        .transform(zscore_within_group)
    )

    for feature in feature_cols:
        z_df[f"{feature}_z"] = (
            z_df
            .groupby(["model", "dataset_label"])[feature]
            .transform(zscore_within_group)
        )

    grand_rows = []

    for model, model_df in z_df.groupby("model"):
        for feature in feature_cols:
            feature_z = f"{feature}_z"

            pearson_r, pearson_p, n_p = safe_corr(
                model_df["auc_z"],
                model_df[feature_z],
                method="pearson",
            )

            spearman_rho, spearman_p, n_s = safe_corr(
                model_df["auc_z"],
                model_df[feature_z],
                method="spearman",
            )

            valid = model_df[
                np.isfinite(model_df["auc_z"])
                & np.isfinite(model_df[feature_z])
                ]

            grand_rows.append({
                "feature_family": family_name,
                "model": model,
                "feature": feature,
                "n": n_p,
                "pearson_r_z": pearson_r,
                "pearson_p_z": pearson_p,
                "spearman_rho_z": spearman_rho,
                "spearman_p_z": spearman_p,
                "abs_pearson_r_z": abs(pearson_r) if np.isfinite(pearson_r) else np.nan,
                "abs_spearman_rho_z": abs(spearman_rho) if np.isfinite(spearman_rho) else np.nan,
                "n_valid_grand": int(len(valid)),
                "n_valid_datasets": int(valid["dataset_label"].nunique()),
                "valid_datasets": ",".join(sorted(valid["dataset_label"].unique())),
            })

    grand_corr_df = pd.DataFrame(grand_rows)

    grand_corr_df = grand_corr_df.sort_values(
        ["model", "abs_spearman_rho_z"],
        ascending=[True, False],
    ).reset_index(drop=True)

    return within_corr_df, grand_corr_df, z_df

In [ ]:
raw_within_corr_df, raw_grand_corr_df, raw_z_df = compute_susceptibility_correlations(
    raw_merged_df,
    feature_cols=raw_feature_cols,
    target_col="auc_mean",
    family_name="raw_erp_power",
)

# Report-level grand filtering:
# require enough valid subject-recordings and at least 3 datasets.
raw_grand_report_df = raw_grand_corr_df[
    (raw_grand_corr_df["n_valid_grand"] >= 40)
    & (raw_grand_corr_df["n_valid_datasets"] >= 3)
    ].copy()

raw_grand_report_df = raw_grand_report_df.sort_values(
    ["model", "abs_spearman_rho_z"],
    ascending=[True, False],
).reset_index(drop=True)

save_dataframe(raw_within_corr_df, TABLE_DIR / "raw_within_dataset_correlations_all_neural_models.csv")
save_dataframe(raw_grand_corr_df, TABLE_DIR / "raw_zscored_grand_correlations_all_neural_models.csv")
save_dataframe(raw_grand_report_df, TABLE_DIR / "raw_zscored_grand_correlations_report_filtered_all_neural_models.csv")
save_dataframe(raw_z_df, TABLE_DIR / "raw_subject_features_zscored_all_neural_models.csv")

display(round_numeric_columns(raw_grand_report_df.groupby("model").head(10), 4))

In [ ]:
ica_within_corr_df, ica_grand_corr_df, ica_z_df = compute_susceptibility_correlations(
    ica_merged_df,
    feature_cols=ica_feature_cols,
    target_col="auc_mean",
    family_name="ica",
)

ica_grand_report_df = ica_grand_corr_df[
    (ica_grand_corr_df["n_valid_grand"] >= 40)
    & (ica_grand_corr_df["n_valid_datasets"] >= 3)
    ].copy()

ica_grand_report_df = ica_grand_report_df.sort_values(
    ["model", "abs_spearman_rho_z"],
    ascending=[True, False],
).reset_index(drop=True)

save_dataframe(ica_within_corr_df, TABLE_DIR / "ica_within_dataset_correlations_all_neural_models.csv")
save_dataframe(ica_grand_corr_df, TABLE_DIR / "ica_zscored_grand_correlations_all_neural_models.csv")
save_dataframe(ica_grand_report_df, TABLE_DIR / "ica_zscored_grand_correlations_report_filtered_all_neural_models.csv")
save_dataframe(ica_z_df, TABLE_DIR / "ica_subject_features_zscored_all_neural_models.csv")

display(round_numeric_columns(ica_grand_report_df.groupby("model").head(10), 4))

In [ ]:
# Raw/ERP/power FDR
raw_within_fdr_df = add_standard_fdr_columns(
    raw_within_corr_df,
    family_cols=["model", "dataset_label"],
    pearson_p_col="pearson_p",
    spearman_p_col="spearman_p",
    pearson_q_col="pearson_q",
    spearman_q_col="spearman_q",
)

raw_grand_fdr_df = add_standard_fdr_columns(
    raw_grand_corr_df,
    family_cols=["model"],
    pearson_p_col="pearson_p_z",
    spearman_p_col="spearman_p_z",
    pearson_q_col="pearson_q_z",
    spearman_q_col="spearman_q_z",
)

raw_grand_report_fdr_df = add_standard_fdr_columns(
    raw_grand_report_df,
    family_cols=["model"],
    pearson_p_col="pearson_p_z",
    spearman_p_col="spearman_p_z",
    pearson_q_col="pearson_q_z",
    spearman_q_col="spearman_q_z",
)

# ICA FDR
ica_within_fdr_df = add_standard_fdr_columns(
    ica_within_corr_df,
    family_cols=["model", "dataset_label"],
    pearson_p_col="pearson_p",
    spearman_p_col="spearman_p",
    pearson_q_col="pearson_q",
    spearman_q_col="spearman_q",
)

ica_grand_fdr_df = add_standard_fdr_columns(
    ica_grand_corr_df,
    family_cols=["model"],
    pearson_p_col="pearson_p_z",
    spearman_p_col="spearman_p_z",
    pearson_q_col="pearson_q_z",
    spearman_q_col="spearman_q_z",
)

ica_grand_report_fdr_df = add_standard_fdr_columns(
    ica_grand_report_df,
    family_cols=["model"],
    pearson_p_col="pearson_p_z",
    spearman_p_col="spearman_p_z",
    pearson_q_col="pearson_q_z",
    spearman_q_col="spearman_q_z",
)

save_dataframe(raw_within_fdr_df, TABLE_DIR / "raw_within_dataset_correlations_all_neural_models_fdr.csv")
save_dataframe(raw_grand_fdr_df, TABLE_DIR / "raw_zscored_grand_correlations_all_neural_models_fdr.csv")
save_dataframe(raw_grand_report_fdr_df, TABLE_DIR / "raw_zscored_grand_correlations_report_filtered_all_neural_models_fdr.csv")

save_dataframe(ica_within_fdr_df, TABLE_DIR / "ica_within_dataset_correlations_all_neural_models_fdr.csv")
save_dataframe(ica_grand_fdr_df, TABLE_DIR / "ica_zscored_grand_correlations_all_neural_models_fdr.csv")
save_dataframe(ica_grand_report_fdr_df, TABLE_DIR / "ica_zscored_grand_correlations_report_filtered_all_neural_models_fdr.csv")

print("FDR correction complete.")

print("\nRaw grand report top q-values:")
display(
    round_numeric_columns(
        raw_grand_report_fdr_df
        .sort_values(["model", "spearman_q_z", "abs_spearman_rho_z"])
        .groupby("model")
        .head(8),
        4,
    )
)

print("\nICA grand report top q-values:")
display(
    round_numeric_columns(
        ica_grand_report_fdr_df
        .sort_values(["model", "spearman_q_z", "abs_spearman_rho_z"])
        .groupby("model")
        .head(8),
        4,
    )
)

In [ ]:
summary_frames = []

summary_frames.append(
    summarize_fdr_table(
        raw_within_fdr_df,
        table_name="raw_within_dataset_all_neural_models",
        family_cols=["model", "dataset_label"],
        q_col="spearman_q",
        p_col="spearman_p",
        effect_col="spearman_rho",
    )
)

summary_frames.append(
    summarize_fdr_table(
        raw_grand_report_fdr_df,
        table_name="raw_grand_report_all_neural_models",
        family_cols=["model"],
        q_col="spearman_q_z",
        p_col="spearman_p_z",
        effect_col="spearman_rho_z",
    )
)

summary_frames.append(
    summarize_fdr_table(
        ica_within_fdr_df,
        table_name="ica_within_dataset_all_neural_models",
        family_cols=["model", "dataset_label"],
        q_col="spearman_q",
        p_col="spearman_p",
        effect_col="spearman_rho",
    )
)

summary_frames.append(
    summarize_fdr_table(
        ica_grand_report_fdr_df,
        table_name="ica_grand_report_all_neural_models",
        family_cols=["model"],
        q_col="spearman_q_z",
        p_col="spearman_p_z",
        effect_col="spearman_rho_z",
    )
)

fdr_summary_df = pd.concat(summary_frames, ignore_index=True)

save_dataframe(
    fdr_summary_df,
    TABLE_DIR / "fdr_summary_by_family_all_neural_models.csv",
)

display(round_numeric_columns(fdr_summary_df, 4))

In [ ]:
def collect_significant_results(q_threshold: float) -> pd.DataFrame:
    tables = []

    specs = [
        (
            raw_grand_report_fdr_df,
            "raw_grand_report_all_neural_models",
            "spearman_q_z",
            "spearman_rho_z",
        ),
        (
            raw_within_fdr_df,
            "raw_within_dataset_all_neural_models",
            "spearman_q",
            "spearman_rho",
        ),
        (
            ica_grand_report_fdr_df,
            "ica_grand_report_all_neural_models",
            "spearman_q_z",
            "spearman_rho_z",
        ),
        (
            ica_within_fdr_df,
            "ica_within_dataset_all_neural_models",
            "spearman_q",
            "spearman_rho",
        ),
    ]

    for df, analysis_name, q_col, effect_col in specs:
        sig = top_fdr_significant(
            df,
            q_col=q_col,
            effect_col=effect_col,
            q_threshold=q_threshold,
            n_top=200,
        )

        if len(sig) > 0:
            sig = sig.copy()
            sig["analysis"] = analysis_name
            sig["q_col_used"] = q_col
            sig["effect_col_used"] = effect_col
            tables.append(sig)

    if len(tables) == 0:
        return pd.DataFrame()

    return pd.concat(tables, ignore_index=True)


all_sig_fdr05_df = collect_significant_results(0.05)
all_sig_fdr10_df = collect_significant_results(0.10)

save_dataframe(
    all_sig_fdr05_df,
    TABLE_DIR / "all_neural_model_susceptibility_spearman_fdr05_results.csv",
)

save_dataframe(
    all_sig_fdr10_df,
    TABLE_DIR / "all_neural_model_susceptibility_spearman_fdr10_results.csv",
)

print("Number of Spearman q <= 0.05 results:", len(all_sig_fdr05_df))
if len(all_sig_fdr05_df) > 0:
    display(round_numeric_columns(all_sig_fdr05_df.head(100), 4))
else:
    print("No Spearman effects survived FDR q <= 0.05.")

print("\nNumber of Spearman q <= 0.10 results:", len(all_sig_fdr10_df))
if len(all_sig_fdr10_df) > 0:
    display(round_numeric_columns(all_sig_fdr10_df.head(100), 4))
else:
    print("No Spearman effects survived FDR q <= 0.10.")

In [ ]:
raw_report_compact = raw_grand_report_fdr_df[[
    "feature_family",
    "model",
    "feature",
    "n",
    "n_valid_grand",
    "n_valid_datasets",
    "valid_datasets",
    "spearman_rho_z",
    "spearman_p_z",
    "spearman_q_z",
    "spearman_q_z_fdr_005",
    "spearman_q_z_fdr_010",
    "pearson_r_z",
    "pearson_p_z",
    "pearson_q_z",
    "pearson_q_z_fdr_005",
    "pearson_q_z_fdr_010",
]].sort_values(["model", "spearman_q_z", "spearman_p_z"])

ica_report_compact = ica_grand_report_fdr_df[[
    "feature_family",
    "model",
    "feature",
    "n",
    "n_valid_grand",
    "n_valid_datasets",
    "valid_datasets",
    "spearman_rho_z",
    "spearman_p_z",
    "spearman_q_z",
    "spearman_q_z_fdr_005",
    "spearman_q_z_fdr_010",
    "pearson_r_z",
    "pearson_p_z",
    "pearson_q_z",
    "pearson_q_z_fdr_005",
    "pearson_q_z_fdr_010",
]].sort_values(["model", "spearman_q_z", "spearman_p_z"])

save_dataframe(
    raw_report_compact,
    TABLE_DIR / "report_raw_grand_correlations_all_neural_models_with_qvalues.csv",
)

save_dataframe(
    ica_report_compact,
    TABLE_DIR / "report_ica_grand_correlations_all_neural_models_with_qvalues.csv",
)

print("Raw grand report top rows:")
display(round_numeric_columns(raw_report_compact.groupby("model").head(8), 4))

print("\nICA grand report top rows:")
display(round_numeric_columns(ica_report_compact.groupby("model").head(8), 4))

In [ ]:
for model, g in raw_grand_report_fdr_df.groupby("model"):
    g = g.copy()
    g = g.sort_values("spearman_q_z", ascending=True).head(15)
    g = g.sort_values("spearman_rho_z", ascending=True)

    fig, ax = plt.subplots(figsize=(9, 6))

    ax.barh(g["feature"], g["spearman_rho_z"])
    ax.axvline(0.0, linewidth=1)

    ax.set_xlabel("Spearman rho with z-scored AUC")
    ax.set_title(f"Raw/ERP/power susceptibility with FDR q-values — {model}")
    ax.grid(axis="x", alpha=0.3)

    for i, (_, row) in enumerate(g.iterrows()):
        ax.text(
            row["spearman_rho_z"],
            i,
            f" q={row['spearman_q_z']:.3f}",
            va="center",
            ha="left" if row["spearman_rho_z"] >= 0 else "right",
            fontsize=8,
        )

    fig.tight_layout()
    save_figure(fig, FIG_DIR / f"raw_grand_fdr_barplot_{model}.png")
    plt.show()

In [ ]:
for model, g in ica_grand_report_fdr_df.groupby("model"):
    g = g.copy()
    g = g.sort_values("spearman_q_z", ascending=True).head(15)
    g = g.sort_values("spearman_rho_z", ascending=True)

    fig, ax = plt.subplots(figsize=(9, 6))

    ax.barh(g["feature"], g["spearman_rho_z"])
    ax.axvline(0.0, linewidth=1)

    ax.set_xlabel("Spearman rho with z-scored AUC")
    ax.set_title(f"ICA susceptibility with FDR q-values — {model}")
    ax.grid(axis="x", alpha=0.3)

    for i, (_, row) in enumerate(g.iterrows()):
        ax.text(
            row["spearman_rho_z"],
            i,
            f" q={row['spearman_q_z']:.3f}",
            va="center",
            ha="left" if row["spearman_rho_z"] >= 0 else "right",
            fontsize=8,
        )

    fig.tight_layout()
    save_figure(fig, FIG_DIR / f"ica_grand_fdr_barplot_{model}.png")
    plt.show()

In [ ]:
top_n = 15

for model in sorted(raw_grand_report_fdr_df["model"].unique()):
    top_features = (
        raw_grand_report_fdr_df[raw_grand_report_fdr_df["model"] == model]
        .sort_values("spearman_q_z")
        .head(top_n)["feature"]
        .tolist()
    )

    heat_df = raw_within_fdr_df[
        (raw_within_fdr_df["model"] == model)
        & (raw_within_fdr_df["feature"].isin(top_features))
        ].copy()

    if len(heat_df) == 0:
        continue

    heat_pivot = heat_df.pivot(
        index="feature",
        columns="dataset_label",
        values="spearman_rho",
    )

    heat_pivot = heat_pivot.loc[[f for f in top_features if f in heat_pivot.index]]

    fig, ax = plt.subplots(figsize=(8, max(5, 0.35 * len(heat_pivot))))

    im = ax.imshow(
        heat_pivot.values,
        aspect="auto",
        vmin=-1,
        vmax=1,
    )

    ax.set_xticks(np.arange(len(heat_pivot.columns)))
    ax.set_xticklabels(heat_pivot.columns, rotation=45, ha="right")

    ax.set_yticks(np.arange(len(heat_pivot.index)))
    ax.set_yticklabels(heat_pivot.index)

    ax.set_title(f"Raw/ERP/power within-dataset correlations — {model}")
    ax.set_xlabel("Dataset")
    ax.set_ylabel("Feature")

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Spearman rho")

    fig.tight_layout()
    save_figure(fig, FIG_DIR / f"raw_within_dataset_heatmap_{model}.png")
    plt.show()

In [ ]:
print("=" * 90)
print("Neural-model susceptibility + FDR interpretation helper")
print("=" * 90)

print("\nPrimary grand raw/ERP/power Spearman q <= 0.05:")
raw_grand_sig = raw_grand_report_fdr_df[
    raw_grand_report_fdr_df["spearman_q_z"] <= 0.05
    ]
if len(raw_grand_sig) == 0:
    print(" - None survived q <= 0.05.")
else:
    display(
        round_numeric_columns(
            raw_grand_sig.sort_values(["model", "spearman_q_z"])[
                [
                    "model",
                    "feature",
                    "spearman_rho_z",
                    "spearman_p_z",
                    "spearman_q_z",
                    "n_valid_grand",
                    "n_valid_datasets",
                ]
            ],
            4,
        )
    )

print("\nPrimary grand ICA Spearman q <= 0.05:")
ica_grand_sig = ica_grand_report_fdr_df[
    ica_grand_report_fdr_df["spearman_q_z"] <= 0.05
    ]
if len(ica_grand_sig) == 0:
    print(" - None survived q <= 0.05.")
else:
    display(
        round_numeric_columns(
            ica_grand_sig.sort_values(["model", "spearman_q_z"])[
                [
                    "model",
                    "feature",
                    "spearman_rho_z",
                    "spearman_p_z",
                    "spearman_q_z",
                    "n_valid_grand",
                    "n_valid_datasets",
                ]
            ],
            4,
        )
    )

print("\nDataset-specific raw/ERP/power Spearman q <= 0.05:")
raw_within_sig = raw_within_fdr_df[
    raw_within_fdr_df["spearman_q"] <= 0.05
    ]
if len(raw_within_sig) == 0:
    print(" - None survived q <= 0.05.")
else:
    display(
        round_numeric_columns(
            raw_within_sig.sort_values(["model", "dataset_label", "spearman_q"])[
                [
                    "model",
                    "dataset_label",
                    "feature",
                    "spearman_rho",
                    "spearman_p",
                    "spearman_q",
                    "n",
                ]
            ].head(80),
            4,
        )
    )

print("\nDataset-specific ICA Spearman q <= 0.05:")
ica_within_sig = ica_within_fdr_df[
    ica_within_fdr_df["spearman_q"] <= 0.05
    ]
if len(ica_within_sig) == 0:
    print(" - None survived q <= 0.05.")
else:
    display(
        round_numeric_columns(
            ica_within_sig.sort_values(["model", "dataset_label", "spearman_q"])[
                [
                    "model",
                    "dataset_label",
                    "feature",
                    "spearman_rho",
                    "spearman_p",
                    "spearman_q",
                    "n",
                ]
            ].head(80),
            4,
        )
    )

In [ ]:
print("Outputs saved under:", OUT_ROOT)

print("\nTables:")
for p in sorted(TABLE_DIR.glob("*.csv")):
    print(" -", p)

print("\nFigures:")
for p in sorted(FIG_DIR.glob("*.png")):
    print(" -", p)